# Bloque 7 — Grafos e IA
## Notebook 01: Agente investigador sobre la red

---

En el notebook anterior construimos el grafo y calculamos métricas. Ahora añadimos un agente de IA encima: en vez de consultar el grafo manualmente, le hacemos **preguntas en lenguaje natural** y el agente decide qué herramientas usar para responderlas.

### Pipeline

```
 Pregunta en texto
       ↓
 [Agente Analista] — qwen2.5:14b
   ├── contactos_directos(numero)
   ├── contactos_comunes(numero1, numero2)
   ├── comunidad_de(numero)
   ├── nodos_puente(comunidad_a, comunidad_b)
   └── perfil_llamadas(numero)
       ↓
 [Agente Redactor] — deepseek-r1:14b
       ↓
  Informe estructurado
```

> **Prerrequisito**: ejecutar `00_construccion_red.ipynb` antes de este notebook para generar `data/grafo.pkl`, `data/nodos.csv` y `data/llamadas.csv`.

In [1]:
# ─── CARGAR EL GRAFO Y LOS DATOS ─────────────────────────────────────────────
import pickle
import pandas as pd
import igraph as ig
from pathlib import Path

DATA = Path('data')

with open(DATA / 'grafo.pkl', 'rb') as f:
    G = pickle.load(f)

df_nodos   = pd.read_csv(DATA / 'nodos.csv')
df_llamadas = pd.read_csv(DATA / 'llamadas.csv')
df_llamadas['timestamp'] = pd.to_datetime(df_llamadas['timestamp'])

# Índice nombre → id de vértice
NOMBRE_IDX = {v['nombre']: v.index for v in G.vs}

print(f'Grafo cargado: {G.vcount():,} nodos | {G.ecount():,} aristas')
print(f'Comunidades  : {df_nodos["comunidad"].nunique()}')
print(f'Llamadas     : {len(df_llamadas):,}')
print(f'\nTeléfonos core:')
for n, g_id, com in df_nodos[df_nodos["tipo"].str.startswith("Grupo")][["nombre", "tipo", "comunidad"]].values:
    print(f'  {n}  {g_id:<10}  comunidad {com}')

Grafo cargado: 540 nodos | 712 aristas
Comunidades  : 3
Llamadas     : 15,000

Teléfonos core:
  34600000001  Grupo A     comunidad 0
  34600000002  Grupo A     comunidad 0
  34600000003  Grupo A     comunidad 0
  34600000004  Grupo A     comunidad 0
  34600000005  Grupo B     comunidad 1
  34600000006  Grupo B     comunidad 1
  34600000007  Grupo B     comunidad 1
  34600000008  Grupo C     comunidad 2
  34600000009  Grupo C     comunidad 2
  34600000010  Grupo C     comunidad 2


In [2]:
# ─── HERRAMIENTAS DE CONSULTA SOBRE EL GRAFO ─────────────────────────────────
from crewai.tools import tool

@tool
def contactos_directos(numero: str) -> str:
    """Lista los números con los que ha llamado este teléfono y cuántas veces, ordenados por frecuencia."""
    if numero not in NOMBRE_IDX:
        return f"Número {numero} no encontrado en el grafo."
    idx = NOMBRE_IDX[numero]
    vecinos = G.neighbors(idx)
    resultado = []
    for v in vecinos:
        eid = G.get_eid(idx, v)
        resultado.append((G.vs[v]['nombre'], G.es[eid]['peso']))
    resultado.sort(key=lambda x: -x[1])
    lineas = [f'  {n}: {int(p)} llamadas' for n, p in resultado[:20]]
    return f"Contactos de {numero} (top {len(lineas)}):\n" + "\n".join(lineas)


@tool
def contactos_comunes(numero1: str, numero2: str) -> str:
    """Devuelve los teléfonos con los que han llamado ambos números a la vez (contactos compartidos)."""
    for n in [numero1, numero2]:
        if n not in NOMBRE_IDX:
            return f"Número {n} no encontrado."
    vec1 = {G.vs[v]['nombre'] for v in G.neighbors(NOMBRE_IDX[numero1])}
    vec2 = {G.vs[v]['nombre'] for v in G.neighbors(NOMBRE_IDX[numero2])}
    comunes = vec1 & vec2 - {numero1, numero2}
    if not comunes:
        return f"No hay contactos comunes entre {numero1} y {numero2}."
    return (f"Contactos comunes entre {numero1} y {numero2} ({len(comunes)} en total):\n" +
            "\n".join(f"  {c}" for c in sorted(comunes)[:20]))


@tool
def comunidad_de(numero: str) -> str:
    """Devuelve la comunidad Leiden a la que pertenece un número y cuántos miembros tiene esa comunidad."""
    if numero not in NOMBRE_IDX:
        return f"Número {numero} no encontrado."
    com = G.vs[NOMBRE_IDX[numero]]['comunidad']
    miembros = [v['nombre'] for v in G.vs if v['comunidad'] == com]
    muestra = ', '.join(miembros[:5])
    return (f"{numero} → comunidad {com} ({len(miembros)} miembros).\n"
            f"Muestra: {muestra}{'...' if len(miembros) > 5 else ''}")


@tool
def nodos_puente(comunidad_a: int, comunidad_b: int) -> str:
    """Encuentra nodos que tienen contactos en dos comunidades distintas (brokers o intermediarios)."""
    brokers = []
    for v in G.vs:
        vecinos_coms = {G.vs[n]['comunidad'] for n in G.neighbors(v.index)}
        if comunidad_a in vecinos_coms and comunidad_b in vecinos_coms:
            brokers.append((v['nombre'], v['comunidad'], G.degree(v.index)))
    if not brokers:
        return f"No hay brokers entre comunidad {comunidad_a} y comunidad {comunidad_b}."
    brokers.sort(key=lambda x: -x[2])
    lineas = [f"  {n} (comunidad {c}, grado {d})" for n, c, d in brokers[:10]]
    return f"Brokers entre comunidades {comunidad_a} y {comunidad_b}:\n" + "\n".join(lineas)


@tool
def perfil_llamadas(numero: str) -> str:
    """Estadísticas de llamadas de un número: total, duración media, hora pico, ratio saliente/entrante."""
    mask = (df_llamadas['origen'] == numero) | (df_llamadas['destino'] == numero)
    sub = df_llamadas[mask]
    if sub.empty:
        return f"No hay llamadas para {numero}."
    hora_pico = sub['timestamp'].dt.hour.value_counts().idxmax()
    salientes = (sub['origen'] == numero).sum()
    entrantes = (sub['destino'] == numero).sum()
    return (
        f"Perfil de {numero}:\n"
        f"  Total llamadas   : {len(sub):,}\n"
        f"  Duración media   : {sub['duracion_s'].mean():.0f}s\n"
        f"  Duración total   : {sub['duracion_s'].sum() / 3600:.1f}h\n"
        f"  Hora pico        : {hora_pico:02d}:00h\n"
        f"  Salientes        : {salientes}\n"
        f"  Entrantes        : {entrantes}"
    )

print('Herramientas definidas: contactos_directos, contactos_comunes, comunidad_de, nodos_puente, perfil_llamadas')

Herramientas definidas: contactos_directos, contactos_comunes, comunidad_de, nodos_puente, perfil_llamadas


In [3]:
# ─── CONFIGURAR AGENTES ───────────────────────────────────────────────────────
from crewai import Agent, Task, Crew, Process, LLM

OLLAMA_URL = 'http://localhost:11434'
llm_analista = LLM(model='ollama/qwen2.5:14b', base_url=OLLAMA_URL)
llm_redactor = LLM(model='ollama/deepseek-r1:14b', base_url=OLLAMA_URL)

HERRAMIENTAS = [
    contactos_directos, contactos_comunes,
    comunidad_de, nodos_puente, perfil_llamadas,
]

agente_analista = Agent(
    role='Analista de redes de comunicación',
    goal='Investigar patrones de llamadas usando las herramientas de consulta sobre el grafo y extraer hechos concretos.',
    backstory=(
        'Eres un analista especializado en redes de comunicación. '
        'Usas las herramientas disponibles para consultar el grafo y presentas '
        'tus hallazgos de forma estructurada y factual.'
    ),
    tools=HERRAMIENTAS,
    llm=llm_analista,
    verbose=True,
)

agente_redactor = Agent(
    role='Redactor de informes de inteligencia',
    goal='Sintetizar el análisis del analista en un informe claro, estructurado y accionable.',
    backstory=(
        'Redactas informes de inteligencia a partir del análisis del investigador. '
        'Eres preciso, objetivo y usas formato estructurado en markdown.'
    ),
    llm=llm_redactor,
    verbose=True,
)

print('Agentes listos: analista (qwen2.5:14b) + redactor (deepseek-r1:14b)')

Agentes listos: analista (qwen2.5:14b) + redactor (deepseek-r1:14b)


In [4]:
# ─── HELPER: lanzar crew sin bloquear el event loop de Jupyter ────────────────
import concurrent.futures

def investigar(pregunta: str, sujeto: str = '') -> str:
    """
    Lanza el crew analista+redactor con la pregunta dada.
    sujeto: contexto adicional (número o números de interés).
    """
    contexto = f"Sujeto(s) de interés: {sujeto}\n\n" if sujeto else ''

    tarea_analisis = Task(
        description=(
            f"{contexto}Pregunta de investigación:\n{pregunta}\n\n"
            "Usa las herramientas disponibles para obtener datos concretos del grafo. "
            "Presenta los hallazgos como una lista de hechos verificados."
        ),
        expected_output='Lista estructurada de hechos: comunidad, contactos clave, rol en la red, métricas relevantes.',
        agent=agente_analista,
    )

    tarea_informe = Task(
        description=(
            "Con los hallazgos del analista, redacta un informe de inteligencia en markdown. "
            "Incluye: Resumen Ejecutivo, Posición en la Red, Contactos Clave, Conclusiones."
        ),
        expected_output='Informe en markdown con secciones: ## Resumen Ejecutivo, ## Posición en la Red, ## Contactos Clave, ## Conclusiones.',
        agent=agente_redactor,
        context=[tarea_analisis],
    )

    crew = Crew(
        agents=[agente_analista, agente_redactor],
        tasks=[tarea_analisis, tarea_informe],
        process=Process.sequential,
        verbose=False,
    )

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
        resultado = ex.submit(crew.kickoff).result(timeout=600)

    return str(resultado)

print('Helper investigar() listo')

Helper investigar() listo


## Demo 1 — Investigar un número sujeto

Pedimos al crew que analice el teléfono `+34600000001` (grupo A): su comunidad, contactos clave y rol en la red.

In [5]:
# ─── DEMO 1: investigar un número ────────────────────────────────────────────
SUJETO = '+34600000001'

informe1 = investigar(
    pregunta=(
        f"Investiga el teléfono {SUJETO}. "
        "Determina: (1) a qué comunidad pertenece, "
        "(2) sus 10 contactos más frecuentes y qué tipo de nodos son, "
        "(3) su perfil de llamadas (horario pico, duración media), "
        "(4) si actúa como puente entre comunidades."
    ),
    sujeto=SUJETO,
)

print(informe1)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de redes de comunicación                                                                       │
│                                                                                                                 │
│  Task: Sujeto(s) de interés: +34600000001                                                                       │
│                                                                                                                 │
│  Pregunta de investigación:                                                                                     │
│  Investiga el teléfono +34600000001. Determina: (1) a qué comunidad pertenece, (2) sus 10 contactos más         │
│  frecuentes y qué tipo de nodos son, (3) su perfil de llamadas (horario pico, duración media), (4) si actúa     │
│  como puente entre comunidades.                                                                                 │
│                                                                                                                 │
│  Usa las herramientas disponibles para obtener datos concretos del grafo. Presenta los hallazgos como una       │
│  lista de hechos verificados.                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de redes de comunicación                                                                       │
│                                                                                                                 │
│  Thought: Thought: First step is to determine which community +34600000001 belongs to.                          │
│                                                                                                                 │
│  Using Tool: comunidad_de                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"numero\": \"+34600000001\"}"                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  +34600000001 → comunidad 0 (218 miembros).                                                                     │
│  Muestra: +34600000001, +34600000002, +34600000003, +34600000004, +34700000001...                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de redes de comunicación                                                                       │
│                                                                                                                 │
│  Thought: Thought: Now I know the community of +34600000001. Next step is to find its most frequent contacts.   │
│                                                                                                                 │
│  Using Tool: contactos_directos                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor de informes de inteligencia                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # Informe de Inteligencia                                                                                      │
│                                                                                                                 │
│  ## Resumen Ejecutivo                                                                                           │
│  El presente informe analiza el comportamiento y las relaciones del número telefónico +34600000001,             │
│  perteneciente a la comunidad 0. Los datos recopilados indican que dicho número tiene contactos frecuentes      │
│  dentro de su misma comunidad pero no muestra actividad alguna en términos de llamadas realizadas o recibidas.  │
│                                                                                                                 │
│  ## Posición en la Red                                                                                          │
│  - **Comunidad**: El número +34600000001 está integrado en la comunidad 0.                                      │
│  - **Rol**: No actúa como nodo puente, ya que no ha establecido conexiones con números pertenecientes a otras   │
│  comunidades.                                                                                                   │
│  - **Inactividad**: El perfil de llamadas del número se encuentra vacío, lo que sugiere ausencia de actividad   │
│  telefónica.                                                                                                    │
│                                                                                                                 │
│  ## Contactos Clave                                                                                             │
│  - **Dentro de la comunidad 0**: El número mantiene contactos frecuentes con otros miembros de su misma         │
│  comunidad.                                                                                                     │
│  - **Fuera de la comunidad 0**: No se ha identificado ningún contacto fuera de su comunidad, lo que refuerza    │
│  la idea de que no actúa como intermediario entre diferentes grupos.                                            │
│                                                                                                                 │
│  ## Conclusiones                                                                                                │
│  1. El número +34600000001 no parece desempeñar un papelactivo en actividades transversales o intercomunales,   │
│  al menos desde el punto de vista de las comunicaciones telefónicas.                                            │
│  2. La ausencia de actividad telefónica (llamadas) podría indicar que este número no es utilizado activamente   │
│  para fines de comunicación o bien que su uso se limita a funciones específicas no detectadas en los datos      │
│  actuales.                                                                                                      │
│  3. Es recomendable realizar un seguimiento adicional del comportamiento de este número y su entorno social     │
│  dentro de la comunidad 0, ya que podría contener información útil para la investigación.                       │
│                                                        

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

Exception in thread Thread-6 (get_input):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/drjekyll/Documentos/umbrella/FearOfTheDark/.venv/lib/python3.12/site-packages/crewai/events/listeners/tracing/utils.py", line 359, in get_input
    response = input().strip().lower()
               ^^^^^^^
  File "/home/drjekyll/Documentos/umbrella/FearOfTheDark/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 1402, in raw_input
    raise StdinNotImplementedError(msg)
IPython.core.error.StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.


```markdown
# Informe de Inteligencia

## Resumen Ejecutivo  
El presente informe analiza el comportamiento y las relaciones del número telefónico +34600000001, perteneciente a la comunidad 0. Los datos recopilados indican que dicho número tiene contactos frecuentes dentro de su misma comunidad pero no muestra actividad alguna en términos de llamadas realizadas o recibidas.

## Posición en la Red  
- **Comunidad**: El número +34600000001 está integrado en la comunidad 0.  
- **Rol**: No actúa como nodo puente, ya que no ha establecido conexiones con números pertenecientes a otras comunidades.  
- **Inactividad**: El perfil de llamadas del número se encuentra vacío, lo que sugiere ausencia de actividad telefónica.

## Contactos Clave  
- **Dentro de la comunidad 0**: El número mantiene contactos frecuentes con otros miembros de su misma comunidad.  
- **Fuera de la comunidad 0**: No se ha identificado ningún contacto fuera de su comunidad, lo que refuerza la idea de que no actúa como in

## Demo 2 — ¿Quién conecta dos grupos?

Preguntamos qué nodos actúan de puente entre la comunidad de los grupos A y B, y qué tienen en común dos números sujeto de grupos distintos.

In [6]:
# ─── DEMO 2: brokers entre grupos ────────────────────────────────────────────
# Averiguar las comunidades de dos sujetos de grupos distintos
com_A = G.vs[NOMBRE_IDX['+34600000001']]['comunidad']
com_B = G.vs[NOMBRE_IDX['+34600000005']]['comunidad']

informe2 = investigar(
    pregunta=(
        f"Identifica qué nodos actúan de puente entre la comunidad {com_A} y la comunidad {com_B}. "
        f"Analiza también qué contactos comparten +34600000001 y +34600000005. "
        "Explica el papel de estos brokers en la red."
    ),
    sujeto=f'+34600000001 (comunidad {com_A}) y +34600000005 (comunidad {com_B})',
)

print(informe2)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de redes de comunicación                                                                       │
│                                                                                                                 │
│  Task: Sujeto(s) de interés: +34600000001 (comunidad 0) y +34600000005 (comunidad 1)                            │
│                                                                                                                 │
│  Pregunta de investigación:                                                                                     │
│  Identifica qué nodos actúan de puente entre la comunidad 0 y la comunidad 1. Analiza también qué contactos     │
│  comparten +34600000001 y +34600000005. Explica el papel de estos brokers en la red.                            │
│                                                                                                                 │
│  Usa las herramientas disponibles para obtener datos concretos del grafo. Presenta los hallazgos como una       │
│  lista de hechos verificados.                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de redes de comunicación                                                                       │
│                                                                                                                 │
│  Thought: Thought: Necesito identificar los nodos que actúan como puente entre las comunidades 0 y 1.           │
│                                                                                                                 │
│  Using Tool: nodos_puente                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"comunidad_a\": 0, \"comunidad_b\": 1}"                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Brokers entre comunidades 0 y 1:                                                                               │
│    +34600000001 (comunidad 0, grado 73)                                                                         │
│    +34600000002 (comunidad 0, grado 73)                                                                         │
│    +34600000003 (comunidad 0, grado 73)                                                                         │
│    +34600000004 (comunidad 0, grado 73)                                                                         │
│    +34600000005 (comunidad 1, grado 72)                                                                         │
│    +34600000006 (comunidad 1, grado 72)                                                                         │
│    +34600000007 (comunidad 1, grado 72)                                                                         │
│    +34600000008 (comunidad 2, grado 72)                                                                         │
│    +34600000009 (comunidad 2, grado 72)                                                                         │
│    +34600000010 (comunidad 2, grado 72)                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de redes de comunicación                                                                       │
│                                                                                                                 │
│  Thought: Thought: Ahora necesito identificar los contactos comunes entre +34600000001 y +34600000005.          │
│                                                                                                                 │
│  Using Tool: contactos_comunes                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor de informes de inteligencia                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # Informe de Inteligencia                                                                                      │
│                                                                                                                 │
│  ## Resumen Ejecutivo                                                                                           │
│  El análisis revela una estructura compleja y altamente conectiva en la red, con nodos clave actuando como      │
│  puentes entre comunidades 0 y 1. Estos nodos no solo facilitan el intercambio de información, sino que         │
│  también actúan como esquinas estratégicas para el establecimiento de relaciones más amplias. Los contactos     │
│  comunes identificados entre los principales nodos puente sugieren un ecosistema de networking potencialmente   │
│  altamente influenciable.                                                                                       │
│                                                                                                                 │
│  ## Posición en la Red                                                                                          │
│  Los nodos puente desempeñan un papel crítico en la red, actuando como桥梁entre las comunidades 0 y 1. Algunos  │
│  ejemplos destacados son:                                                                                       │
│  - **Comunidad 0**: +34600000001, +34600000002, +34600000003, +34600000004.                                     │
│  - **Comunidades 1 y 2**: +34600000005, +34600000006, +34600000007.                                             │
│                                                                                                                 │
│  Estos nodos no solo conectan a individuos de comunidades diferentes, sino que también tienen altos grados de   │
│  conexión, lo que los convierte en actores centrales en la red. Por ejemplo, el nodo +34600000001 tiene un      │
│  grado de 73, indicando una alta actividad y relevancia.                                                        │
│                                                                                                                 │
│  ## Contactos Clave                                                                                             │
│  - **Nodos Puente Compartidos**: Los nodos +34600000001 a +34600000004 en comunidad 0, y +34600000005 a         │
│  +34600000007 en comunidades 1 y 2, actúan como puentes críticos.                                               │
│  - **Contactos Comunes**: Los nodos +34700000001 a +34700000010 son contactos comunes entre +34600000001 y      │
│  +34600000005, formando una base sólida para relaciones estratégicas en la red.                                 │
│                                                                                                                 │
│  ## Conclusiones                                                                                                │
│  La estructura de la red, con nodos puente y contactos comunes bien definidos, sugiere una red altamente        │
│  organizada y eficiente. Los nodos clave identificados son vulnerables a tácticas de influencia o               │
│  contra-inteligencia. Es recomendable priorizar estos pun

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

Exception in thread Thread-7 (get_input):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/drjekyll/Documentos/umbrella/FearOfTheDark/.venv/lib/python3.12/site-packages/crewai/events/listeners/tracing/utils.py", line 359, in get_input
    response = input().strip().lower()
               ^^^^^^^
  File "/home/drjekyll/Documentos/umbrella/FearOfTheDark/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 1402, in raw_input
    raise StdinNotImplementedError(msg)
IPython.core.error.StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.


```markdown
# Informe de Inteligencia

## Resumen Ejecutivo  
El análisis revela una estructura compleja y altamente conectiva en la red, con nodos clave actuando como puentes entre comunidades 0 y 1. Estos nodos no solo facilitan el intercambio de información, sino que también actúan como esquinas estratégicas para el establecimiento de relaciones más amplias. Los contactos comunes identificados entre los principales nodos puente sugieren un ecosistema de networking potencialmente altamente influenciable.

## Posición en la Red  
Los nodos puente desempeñan un papel crítico en la red, actuando como桥梁entre las comunidades 0 y 1. Algunos ejemplos destacados son:  
- **Comunidad 0**: +34600000001, +34600000002, +34600000003, +34600000004.  
- **Comunidades 1 y 2**: +34600000005, +34600000006, +34600000007.  

Estos nodos no solo conectan a individuos de comunidades diferentes, sino que también tienen altos grados de conexión, lo que los convierte en actores centrales en la red. Por ejemp

## Demo 3 — ¿Qué tienen en común tres números?

Dados tres números de grupos distintos, el crew busca si hay contactos comunes y qué dice eso sobre sus relaciones.

In [7]:
# ─── DEMO 3: red de tres números ─────────────────────────────────────────────
TRES = ['+34600000002', '+34600000006', '+34600000009']

informe3 = investigar(
    pregunta=(
        f"Analiza los tres teléfonos: {', '.join(TRES)}. "
        "Para cada par, identifica si tienen contactos comunes. "
        "¿Pertenecen a la misma comunidad o a comunidades distintas? "
        "¿Existe algún intermediario que los conecte a todos?"
    ),
    sujeto=', '.join(TRES),
)

print(informe3)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de redes de comunicación                                                                       │
│                                                                                                                 │
│  Task: Sujeto(s) de interés: +34600000002, +34600000006, +34600000009                                           │
│                                                                                                                 │
│  Pregunta de investigación:                                                                                     │
│  Analiza los tres teléfonos: +34600000002, +34600000006, +34600000009. Para cada par, identifica si tienen      │
│  contactos comunes. ¿Pertenecen a la misma comunidad o a comunidades distintas? ¿Existe algún intermediario     │
│  que los conecte a todos?                                                                                       │
│                                                                                                                 │
│  Usa las herramientas disponibles para obtener datos concretos del grafo. Presenta los hallazgos como una       │
│  lista de hechos verificados.                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de redes de comunicación                                                                       │
│                                                                                                                 │
│  Thought: Thought: Primero, necesito analizar cada número individualmente para entender sus comunidades y       │
│  contactos directos.                                                                                            │
│                                                                                                                 │
│  Using Tool: comunidad_de                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"numero\": \"+34600000002\"}"                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  +34600000002 → comunidad 0 (218 miembros).                                                                     │
│  Muestra: +34600000001, +34600000002, +34600000003, +34600000004, +34700000001...                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de redes de comunicación                                                                       │
│                                                                                                                 │
│  Thought: Thought: Ahora que sé a qué comunidad pertenece +34600000002, analizaré los otros dos números.        │
│                                                                                                                 │
│  Using Tool: comunidad_de                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor de informes de inteligencia                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Resumen Ejecutivo                                                                                           │
│  El análisis revela la estructura de una red telefónica compuesta por tres comunidades principales y varios     │
│  nodos que actúan como intermediarios. Los números telefónicos analizados se encuentran distribuidos en estas   │
│  comunidades, con un total de 73 miembros en la comunidad 0 y 72 en las comunidades 1 y 2 respectivamente.      │
│  Aunque estos números no comparten contactos directos, existen nodos clave que actúan como "brokers" entre las  │
│  diferentes comunidades.                                                                                        │
│                                                                                                                 │
│  ## Posición en la Red                                                                                          │
│  - **Comunidad 0 (+34600000002):** Ubicado en una comunidad con 73 miembros, este nodo no comparte contactos    │
│  directos con los números de las otras comunidades.                                                             │
│  - **Comunidades 1 y 2 (+34600000006 y +34600000009):** Ambos números se encuentran en comunidades más          │
│  pequeñas, pero tampoco comparten contactos directos entre sí.                                                  │
│                                                                                                                 │
│  ## Contactos Clave                                                                                             │
│  - **Nodos Puente:** Los nodos +34600000001 y +34600000002 actúan como intermediarios clave en la red. Estos    │
│  nodos conectan a las diferentes comunidades, facilitando la comunicación entre los mismos.                     │
│                                                                                                                 │
│  ## Conclusiones                                                                                                │
│  La red telefónica analizada muestra una estructura compleja con tres comunidades principales y varios nodos    │
│  que actúan como "brokers" para asegurar la interconexión entre ellas. Los contactos clave identificados        │
│  (+34600000001 y +34600000002) son fundamentales para comprender la dinámica de la red y su capacidad para      │
│  mantener las conexiones entre los diferentes grupos.                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

Exception in thread Thread-8 (get_input):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/drjekyll/Documentos/umbrella/FearOfTheDark/.venv/lib/python3.12/site-packages/crewai/events/listeners/tracing/utils.py", line 359, in get_input
    response = input().strip().lower()
               ^^^^^^^
  File "/home/drjekyll/Documentos/umbrella/FearOfTheDark/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 1402, in raw_input
    raise StdinNotImplementedError(msg)
IPython.core.error.StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.


## Resumen Ejecutivo
El análisis revela la estructura de una red telefónica compuesta por tres comunidades principales y varios nodos que actúan como intermediarios. Los números telefónicos analizados se encuentran distribuidos en estas comunidades, con un total de 73 miembros en la comunidad 0 y 72 en las comunidades 1 y 2 respectivamente. Aunque estos números no comparten contactos directos, existen nodos clave que actúan como "brokers" entre las diferentes comunidades.

## Posición en la Red
- **Comunidad 0 (+34600000002):** Ubicado en una comunidad con 73 miembros, este nodo no comparte contactos directos con los números de las otras comunidades.
- **Comunidades 1 y 2 (+34600000006 y +34600000009):** Ambos números se encuentran en comunidades más pequeñas, pero tampoco comparten contactos directos entre sí.

## Contactos Clave
- **Nodos Puente:** Los nodos +34600000001 y +34600000002 actúan como intermediarios clave en la red. Estos nodos conectan a las diferentes comunidades, faci